In this tutorial, we are going to evaluate the performance of the naive RAG and the GraphRAG algorithm on a [multi-hop RAG task](https://github.com/yixuantt/MultiHop-RAG).

## Setup
Make sure you install the necessary dependencies by running the following commands:

Import the necessary libraries, and set up your openai api key if needed:

In [1]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [2]:
#os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY"
import json
import sys
sys.path.append("../..")

import nest_asyncio
nest_asyncio.apply()
import logging

logging.basicConfig(level=logging.WARNING)
logging.getLogger("nano-graphrag").setLevel(logging.INFO)
from nano_graphrag import GraphRAG, QueryParam
from datasets import Dataset 
from ragas import evaluate
from ragas.metrics import (
    answer_correctness,
    answer_similarity,
)

c:\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Download the dataset from [Github Repo](https://github.com/yixuantt/MultiHop-RAG/tree/main/dataset). 
If should contain two files:
- `MultiHopRAG.json`
- `corpus.json`

After downloading the dataset, replace the below paths to the paths on your machine.

In [3]:

multi_hop_rag_file = "./fixtures/MultiHopRAG.json"
multi_hop_corpus_file = "./fixtures/corpus.json"

## Preprocess

In [4]:

with open(multi_hop_rag_file) as f:
    multi_hop_rag_dataset = json.load(f)
with open(multi_hop_corpus_file) as f:
    multi_hop_corpus = json.load(f)

corups_url_refernces = {}
for cor in multi_hop_corpus:
    corups_url_refernces[cor['url']] = cor

We only use the top-100 queries for evaluation.

In [6]:
'''
multi_hop_rag_dataset = multi_hop_rag_dataset[:100]
print("Queries have types:", set([q['question_type'] for q in multi_hop_rag_dataset]))
total_urls = set()
for q in multi_hop_rag_dataset:
    total_urls.update([up['url'] for up in q['evidence_list']])
corups_url_refernces = {k:v for k, v in corups_url_refernces.items() if k in total_urls}

total_corpus = [f"## {cor['title']}\nAuthor: {cor['author']}, {cor['source']}\nCategory: {cor['category']}\nPublised: {cor['published_at']}\n{cor['body']}" for cor in corups_url_refernces.values()]

print(f"We will need {len(total_corpus)} articles:")
print(total_corpus[0][:200], "...")
'''
import pandas as pd
# Load toy datasets
documents_df = pd.read_csv("./fixtures/toydataset/documents.csv")
queries_df = pd.read_csv("./fixtures/toydataset/multi_passage_answer_questions.csv")

# Prepare corpus
total_corpus = documents_df['text'].tolist()

Add index for the `total_corups` using naive RAG and GraphRAG

In [7]:
# First time indexing will cost many time, roughly 15~20 minutes
from nano_graphrag._llm import openai_complete_if_cache  # 添加这行
from openai import AsyncOpenAI
from typing import Optional, List
from nano_graphrag._utils import compute_args_hash
# openrouter
# 配置OpenRouter客户端
deepseek_client = AsyncOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    default_headers={
        "HTTP-Referer": "https://your-domain.com",  # 替换为实际域名
        "X-Title": "MultiHop RAG Evaluation"         # 项目名称
    }
)

async def openrouter_deepseek_wrapper(
    prompt: str, 
    system_prompt: Optional[str] = None,
    history_messages: List[dict] = [],
    **kwargs
) -> str:
    # 使用独立客户端实例
    client = AsyncOpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.getenv("OPENROUTER_API_KEY"),
        default_headers={
            "HTTP-Referer": "https://your-domain.com",
            "X-Title": "MultiHop RAG Evaluation"
        }
    )
    
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.extend(history_messages)
    messages.append({"role": "user", "content": prompt})
    
    # 缓存处理
    hashing_kv = kwargs.pop("hashing_kv", None)
    if hashing_kv:
        args_hash = compute_args_hash("deepseek/deepseek-chat:free", messages)
        cached = await hashing_kv.get_by_id(args_hash)
        if cached: return cached["return"]

    # 直接调用API
    response = await client.chat.completions.create(
        model="deepseek/deepseek-chat:free",
        messages=messages,
        **kwargs
    )
    # 添加空值校验
    if not response or not response.choices:
        raise ValueError("Empty response from API")
    
    # 添加类型校验
    first_choice = response.choices[0]
    if not hasattr(first_choice, 'message') or not hasattr(first_choice.message, 'content'):
        raise ValueError("Invalid response format")
    
    # 更新缓存
    result = first_choice.message.content
    if hashing_kv:
        await hashing_kv.upsert({
            args_hash: {"return": result, "model": "deepseek/deepseek-chat:free"}
        })
    
    return result
test_response = await openrouter_deepseek_wrapper("Hello")
print(test_response)  
# 正确初始化GraphRAG
graphrag_func = GraphRAG(
    working_dir="nano_graphrag_cache_multi_hop_rag_test",
    enable_naive_rag=True,
    embedding_func_max_async=2,
    embedding_batch_num=64,
    best_model_func=openrouter_deepseek_wrapper,
    cheap_model_func=openrouter_deepseek_wrapper
)

'''
# 在cell 13的初始化前添加DeepSeek配置
deepseek_client = AsyncOpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com/v1"
)

def deepseek_complete_wrapper(*args, **kwargs):
    return openai_complete_if_cache(
        *args, 
        model="deepseek-chat",  # 指定模型
        openai_async_client=deepseek_client,  # 注入自定义客户端
        **kwargs
    )

# 修改原初始化代码
graphrag_func = GraphRAG(
    working_dir="nano_graphrag_cache_multi_hop_rag_test",
    enable_naive_rag=True,
    embedding_func_max_async=2,
    embedding_batch_num=64,
    best_model_func=deepseek_complete_wrapper,  # 使用DeepSeek
    cheap_model_func=deepseek_complete_wrapper   # 双模型都用DeepSeek
)
'''

'''
# using openai api
graphrag_func = GraphRAG(working_dir="nano_graphrag_cache_multi_hop_rag_test", 
                         enable_naive_rag=True,
                         embedding_func_max_async=2,
                         embedding_batch_num=64)
                         '''

graphrag_func.insert(total_corpus)

INFO:nano-graphrag:Load KV full_docs with 0 data
INFO:nano-graphrag:Load KV text_chunks with 0 data
INFO:nano-graphrag:Load KV llm_response_cache with 238 data
INFO:nano-graphrag:Load KV community_reports with 0 data
INFO:nano-graphrag:Loaded graph from nano_graphrag_cache_multi_hop_rag_test\graph_chunk_entity_relation.graphml with 0 nodes, 0 edges
INFO:nano-graphrag:[New Docs] inserting 20 docs


INFO:nano-graphrag:[New Chunks] inserting 155 chunks
INFO:nano-graphrag:Insert chunks for naive RAG
INFO:nano-graphrag:Inserting 155 vectors to chunks
INFO:nano-graphrag:[Entity Extraction]...
INFO:nano-graphrag:Writing graph with 0 nodes, 0 edges


ValueError: Empty response from API

Look at the response of different RAG methods on the first query:

In [24]:
response_formate = "Single phrase or sentence, concise and no redundant explanation needed. If you don't have the answer in context, Just response 'Insufficient information'"
naive_rag_query_param = QueryParam(mode='naive', response_type=response_formate)
naive_rag_query_only_context_param = QueryParam(mode='naive', only_need_context=True)
local_graphrag_query_param = QueryParam(mode='local', response_type=response_formate)
local_graphrag_only_context__param = QueryParam(mode='local', only_need_context=True)

In [8]:
query = multi_hop_rag_dataset[0]
print("Question:", query['query'])
print("GroundTruth Answer:", query['answer'])

Question: Who is the individual associated with the cryptocurrency industry facing a criminal trial on fraud and conspiracy charges, as reported by both The Verge and TechCrunch, and is accused by prosecutors of committing fraud for personal gain?
GroundTruth Answer: Sam Bankman-Fried


In [9]:
print("NaiveRAG Answer:", graphrag_func.query(query['query'], param=naive_rag_query_param))

INFO:nano-graphrag:Truncate 20 to 12 chunks


NaiveRAG Answer: Sam Bankman-Fried


In [10]:
print("Local GraphRAG Answer:", graphrag_func.query(query['query'], param=local_graphrag_query_param))

INFO:nano-graphrag:Using 20 entites, 3 communities, 124 relations, 3 text units


Local GraphRAG Answer: Sam Bankman-Fried


Great! Now we're ready to evaluate more detailed metrics. We will use [ragas](https://docs.ragas.io/en/stable/) to evalue the answers' quality.

In [11]:
questions = [q['query'] for q in multi_hop_rag_dataset]
labels = [q['answer'] for q in multi_hop_rag_dataset]

In [12]:
from tqdm import tqdm
logging.getLogger("nano-graphrag").setLevel(logging.WARNING)

naive_rag_answers = [
    graphrag_func.query(q, param=naive_rag_query_param) for q in tqdm(questions)
]

  0%|          | 0/100 [00:00<?, ?it/s]

100%|██████████| 100/100 [03:53<00:00,  2.33s/it]


In [14]:
local_graphrag_answers = [
    graphrag_func.query(q, param=local_graphrag_query_param) for q in tqdm(questions)
]

100%|██████████| 100/100 [09:10<00:00,  5.50s/it]


In [34]:
naive_results = evaluate(
    Dataset.from_dict({
        "question": questions,
        "ground_truth": labels,
        "answer": naive_rag_answers,
    }),
    metrics=[
        # answer_relevancy,
        answer_correctness,
        answer_similarity,
    ],
)

Evaluating: 100%|██████████| 200/200 [00:32<00:00,  6.19it/s]


In [36]:
local_graphrag_results = evaluate(
    Dataset.from_dict({
        "question": questions,
        "ground_truth": labels,
        "answer": local_graphrag_answers,
    }),
    metrics=[
        # answer_relevancy,
        answer_correctness,
        answer_similarity,
    ],
)

Evaluating: 100%|██████████| 200/200 [00:23<00:00,  8.59it/s]


In [39]:
print("Naive RAG results", naive_results)
print("Local GraphRAG results", local_graphrag_results)

Naive RAG results {'answer_correctness': 0.5896, 'answer_similarity': 0.8935}
Local GraphRAG results {'answer_correctness': 0.7380, 'answer_similarity': 0.8619}
